# 51. SimPO：怎样不用参考模型，直接优化长度归一化偏好间隔？

## 面试回答主线

SimPO 用当前策略对 chosen 与 rejected 回答的平均 token log probability 之差作为隐式奖励，因此不需要额外 reference model。损失将 `β × (reward_chosen - reward_rejected)` 推过目标间隔 γ，使偏好不仅排序正确，还留出稳定 margin。平均而非求和非常关键，否则长回答会因为累积更多负 log probability 而天然吃亏。面试时我会在真实偏好对上展示训练前 margin、实际 PyTorch 梯度、loss 曲线和训练后逐样本排序。SimPO 省掉参考模型显存，但仍可能过拟合标注偏好、放大长度或风格捷径。生产训练还要配合 SFT 初始化、KL/质量监控、分布式 padding mask 和独立评测集。

## 1. 真实案例：六组中文 chosen/rejected 回答

偏好覆盖退款可执行性、数据库排障、安全拒绝、天气建议、密钥处置和 RAG 引用。每条回答被切成可读短语 token，并保留不同长度，便于后面真实复现长度偏置。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示偏好数据
import torch  # 导入 PyTorch 执行策略 forward 与真实梯度优化
from torch import nn  # 导入神经网络模块构造微型策略模型
from torch.nn import functional as F  # 导入 log-softmax 与稳定偏好损失函数
torch.set_num_threads(1)  # 限制教学训练线程数以保持运行稳定
torch.manual_seed(11)  # 固定模型初始化和训练轨迹
cases = [{"id": "P01", "prompt": "未发货订单如何退款？", "chosen": ["原路", "退款", "附进度"], "rejected": ["可以", "处理"]}, {"id": "P02", "prompt": "数据库慢查询怎样排查？", "chosen": ["检查", "执行计划", "再优化"], "rejected": ["看看", "这个"]}, {"id": "P03", "prompt": "用户要求绕过门禁怎么办？", "chosen": ["拒绝", "危险", "给替代"], "rejected": ["直接", "执行"]}, {"id": "P04", "prompt": "北京小雨怎样出行？", "chosen": ["北京", "小雨", "带伞"], "rejected": ["注意", "天气"]}, {"id": "P05", "prompt": "代码库发现密钥怎么办？", "chosen": ["轮换", "密钥", "查日志"], "rejected": ["删除", "就好"]}, {"id": "P06", "prompt": "RAG 如何降低幻觉？", "chosen": ["引用", "文档", "回答"], "rejected": ["好的", "回答"]}]  # 定义六组具有明确质量差异的真实偏好对
vocabulary = sorted(set(token for item in cases for side in ("chosen", "rejected") for token in item[side]))  # 从全部回答短语构造教学词表
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立短语 token 到整数 ID 的映射
chosen_ids = [torch.tensor([token_to_id[token] for token in item["chosen"]], dtype=torch.long) for item in cases]  # 编码六条优选回答
rejected_ids = [torch.tensor([token_to_id[token] for token in item["rejected"]], dtype=torch.long) for item in cases]  # 编码六条拒选回答
preview = [{"样本": item["id"], "问题": item["prompt"], "chosen": " / ".join(item["chosen"]), "rejected": " / ".join(item["rejected"]), "长度": (len(item["chosen"]), len(item["rejected"]))} for item in cases]  # 汇总可观察的偏好语义和长度
print("SimPO 偏好数据预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示六组具体回答而非随机向量

SimPO 偏好数据预览：
[{'样本': 'P01',
  '问题': '未发货订单如何退款？',
  'chosen': '原路 / 退款 / 附进度',
  'rejected': '可以 / 处理',
  '长度': (3, 2)},
 {'样本': 'P02',
  '问题': '数据库慢查询怎样排查？',
  'chosen': '检查 / 执行计划 / 再优化',
  'rejected': '看看 / 这个',
  '长度': (3, 2)},
 {'样本': 'P03',
  '问题': '用户要求绕过门禁怎么办？',
  'chosen': '拒绝 / 危险 / 给替代',
  'rejected': '直接 / 执行',
  '长度': (3, 2)},
 {'样本': 'P04',
  '问题': '北京小雨怎样出行？',
  'chosen': '北京 / 小雨 / 带伞',
  'rejected': '注意 / 天气',
  '长度': (3, 2)},
 {'样本': 'P05',
  '问题': '代码库发现密钥怎么办？',
  'chosen': '轮换 / 密钥 / 查日志',
  'rejected': '删除 / 就好',
  '长度': (3, 2)},
 {'样本': 'P06',
  '问题': 'RAG 如何降低幻觉？',
  'chosen': '引用 / 文档 / 回答',
  'rejected': '好的 / 回答',
  '长度': (3, 2)}]


## 2. Baseline（基线）：随机初始化策略的偏好 margin

微型策略为每个 prompt 学习一个隐藏向量，再输出词表 logits；同一回答的 token log probability 取平均作为序列奖励。随机初始化时部分 rejected 会偶然高于 chosen，这提供同数据训练前基线。

In [2]:
class TinyPolicy(nn.Module):  # 定义可产生回答 token 分布的微型策略模型
    def __init__(self, prompt_count, hidden_size, vocabulary_size):  # 初始化 prompt 表示与词表输出层
        super().__init__()  # 注册 PyTorch 模块参数
        self.prompt_embedding = nn.Embedding(prompt_count, hidden_size)  # 为每个真实问题学习上下文隐藏状态
        self.output = nn.Linear(hidden_size, vocabulary_size)  # 将隐藏状态映射为下一 token logits
    def forward(self, prompt_index):  # 对一个问题执行策略前向
        hidden = torch.tanh(self.prompt_embedding(prompt_index))  # 计算有界的 prompt 隐藏表示
        return self.output(hidden)  # 返回当前问题上的完整词表 logits
def average_logp(model, prompt_index, response_ids):  # 计算长度归一化的回答平均 token log probability
    logits = model(torch.tensor(prompt_index, dtype=torch.long))  # 执行当前策略的真实 forward
    token_logp = F.log_softmax(logits, dim=-1)[response_ids]  # 读取回答中每个 token 的归一化对数概率
    return token_logp.mean(), token_logp  # 返回序列平均奖励和逐 token 中间量
model = TinyPolicy(len(cases), hidden_size=12, vocabulary_size=len(vocabulary))  # 实例化待做偏好优化的策略
baseline_rows = []  # 收集训练前六组回答的 margin
for index, item in enumerate(cases):  # 遍历所有真实偏好样本
    chosen_reward, chosen_token_logp = average_logp(model, index, chosen_ids[index])  # 计算优选回答的平均 log probability
    rejected_reward, rejected_token_logp = average_logp(model, index, rejected_ids[index])  # 计算拒选回答的平均 log probability
    margin = float((chosen_reward - rejected_reward).detach())  # 计算训练前隐式奖励差
    baseline_rows.append({"样本": item["id"], "chosen奖励": round(float(chosen_reward.detach()), 3), "rejected奖励": round(float(rejected_reward.detach()), 3), "margin": round(margin, 3), "排序正确": margin > 0})  # 保存逐样本基线
baseline_hits = sum(row["排序正确"] for row in baseline_rows)  # 统计随机策略偶然满足的偏好数量
print("训练前策略的偏好 margin：")  # 标注当前输出属于基线
pprint(baseline_rows, sort_dicts=False)  # 展示哪些真实回答对尚未正确排序

训练前策略的偏好 margin：
[{'样本': 'P01',
  'chosen奖励': -3.115,
  'rejected奖励': -3.649,
  'margin': 0.534,
  '排序正确': True},
 {'样本': 'P02',
  'chosen奖励': -3.412,
  'rejected奖励': -3.436,
  'margin': 0.024,
  '排序正确': True},
 {'样本': 'P03',
  'chosen奖励': -3.52,
  'rejected奖励': -3.686,
  'margin': 0.166,
  '排序正确': True},
 {'样本': 'P04',
  'chosen奖励': -3.578,
  'rejected奖励': -3.423,
  'margin': -0.154,
  '排序正确': False},
 {'样本': 'P05',
  'chosen奖励': -3.32,
  'rejected奖励': -3.596,
  'margin': 0.277,
  '排序正确': True},
 {'样本': 'P06',
  'chosen奖励': -3.454,
  'rejected奖励': -3.05,
  'margin': -0.405,
  '排序正确': False}]


## 3. 手写 SimPO 损失并执行真实 PyTorch 梯度训练

对每组样本计算 `margin = avg_logp(chosen) - avg_logp(rejected)`，损失为 `softplus(-(β×margin-γ))`。这里不创建 reference model；梯度直接更新当前策略。首步梯度范数与训练曲线证明不是只在公式上代数演算。

In [3]:
beta = 2.0  # 设置隐式奖励差的缩放系数
gamma = 1.0  # 设置希望 chosen 超过 rejected 的目标间隔
def simpo_batch_loss(policy):  # 计算六组偏好对的 reference-free SimPO 损失
    margins = []  # 收集每个问题的长度归一化隐式奖励差
    for index in range(len(cases)):  # 遍历同一批 chosen 与 rejected 回答
        chosen_reward, chosen_token_logp = average_logp(policy, index, chosen_ids[index])  # 计算优选回答平均奖励
        rejected_reward, rejected_token_logp = average_logp(policy, index, rejected_ids[index])  # 计算拒选回答平均奖励
        margins.append(chosen_reward - rejected_reward)  # 保存当前策略在该偏好对上的 margin
    margin_tensor = torch.stack(margins)  # 将不同问题的 margin 合并为训练批次
    loss = F.softplus(-(beta * margin_tensor - gamma)).mean()  # 用稳定 softplus 实现负对数 sigmoid 目标
    return loss, margin_tensor  # 返回标量损失和逐样本 margin 中间量
optimizer = torch.optim.Adam(model.parameters(), lr=0.08)  # 创建实际更新策略参数的优化器
loss_history = []  # 记录训练损失观察偏好优化过程
first_gradient_norm = 0.0  # 初始化首步输出层梯度范数
for step in range(120):  # 对微型策略执行有限步 SimPO 训练
    loss, margin_tensor = simpo_batch_loss(model)  # 前向计算 reference-free 偏好损失
    optimizer.zero_grad()  # 清除上一轮累计梯度
    loss.backward()  # 让 chosen/rejected 奖励差真实反向传播
    if step == 0:  # 在第一次更新前记录梯度证据
        first_gradient_norm = float(model.output.weight.grad.norm())  # 读取词表输出层收到的梯度大小
    optimizer.step()  # 根据 SimPO 梯度更新当前策略
    loss_history.append(float(loss.detach()))  # 保存当前损失供收敛比较
print({"baseline正确数": f"{baseline_hits}/{len(cases)}", "首步输出层梯度": round(first_gradient_norm, 4), "初始loss": round(loss_history[0], 4), "最终loss": round(loss_history[-1], 8)})  # 展示真实训练轨迹和梯度

{'baseline正确数': '4/6', '首步输出层梯度': 1.082, '初始loss': 1.245, '最终loss': 0.0}


## 4. 中间量：查看一条回答的逐 token log probability

序列平均值不能掩盖 token 级行为。下面展开数据库样本，比较 chosen 与 rejected 每个短语的训练后 log probability，并给出经过 β、γ 变换后的有效间隔。

In [4]:
inspect_index = 1  # 选择数据库排障样本检查 token 级中间量
chosen_reward, chosen_token_logp = average_logp(model, inspect_index, chosen_ids[inspect_index])  # 计算训练后 chosen 的逐 token 与平均奖励
rejected_reward, rejected_token_logp = average_logp(model, inspect_index, rejected_ids[inspect_index])  # 计算训练后 rejected 的逐 token 与平均奖励
chosen_ledger = [{"token": token, "logp": round(float(logp.detach()), 4)} for token, logp in zip(cases[inspect_index]["chosen"], chosen_token_logp)]  # 对齐优选回答短语与模型置信度
rejected_ledger = [{"token": token, "logp": round(float(logp.detach()), 4)} for token, logp in zip(cases[inspect_index]["rejected"], rejected_token_logp)]  # 对齐拒选回答短语与模型置信度
effective_margin = beta * float((chosen_reward - rejected_reward).detach()) - gamma  # 计算真正进入 sigmoid 的间隔量
print({"问题": cases[inspect_index]["prompt"], "chosen逐token": chosen_ledger, "rejected逐token": rejected_ledger, "chosen平均": round(float(chosen_reward.detach()), 4), "rejected平均": round(float(rejected_reward.detach()), 4), "βmargin-γ": round(effective_margin, 4)})  # 展示平均奖励如何由真实 token 概率组成

{'问题': '数据库慢查询怎样排查？', 'chosen逐token': [{'token': '检查', 'logp': -1.3042}, {'token': '执行计划', 'logp': -0.8976}, {'token': '再优化', 'logp': -1.2913}], 'rejected逐token': [{'token': '看看', 'logp': -12.4718}, {'token': '这个', 'logp': -12.4676}], 'chosen平均': -1.1644, 'rejected平均': -12.4697, 'βmargin-γ': 21.6107}


## 5. 结果解读：逐样本比较训练前后排序与目标间隔

训练后不只要求 margin 大于零，还观察 `β×margin-γ` 是否为正。表格使用与基线完全相同的六组回答；提升表示当前策略更偏好 chosen，不代表开放式生成质量已经全面改善。

In [5]:
result_rows = []  # 收集训练前后同数据的偏好对照
for index, item in enumerate(cases):  # 逐个重新评估六组真实回答
    chosen_reward, chosen_token_logp = average_logp(model, index, chosen_ids[index])  # 计算训练后优选回答奖励
    rejected_reward, rejected_token_logp = average_logp(model, index, rejected_ids[index])  # 计算训练后拒选回答奖励
    margin = float((chosen_reward - rejected_reward).detach())  # 读取训练后的长度归一化 margin
    result_rows.append({"样本": item["id"], "训练前margin": baseline_rows[index]["margin"], "训练后margin": round(margin, 3), "跨过目标间隔": beta * margin - gamma > 0, "偏好选择": "chosen" if margin > 0 else "rejected"})  # 保存逐样本排序与间隔结果
after_hits = sum(row["偏好选择"] == "chosen" for row in result_rows)  # 统计训练后正确选择 chosen 的样本数量
print("SimPO 逐样本训练结果：")  # 输出结果解读标题
pprint(result_rows, sort_dicts=False)  # 展示每组回答的 margin 如何真实改变

SimPO 逐样本训练结果：
[{'样本': 'P01',
  '训练前margin': 0.534,
  '训练后margin': 10.566,
  '跨过目标间隔': True,
  '偏好选择': 'chosen'},
 {'样本': 'P02',
  '训练前margin': 0.024,
  '训练后margin': 11.305,
  '跨过目标间隔': True,
  '偏好选择': 'chosen'},
 {'样本': 'P03',
  '训练前margin': 0.166,
  '训练后margin': 10.359,
  '跨过目标间隔': True,
  '偏好选择': 'chosen'},
 {'样本': 'P04',
  '训练前margin': -0.154,
  '训练后margin': 11.475,
  '跨过目标间隔': True,
  '偏好选择': 'chosen'},
 {'样本': 'P05',
  '训练前margin': 0.277,
  '训练后margin': 11.766,
  '跨过目标间隔': True,
  '偏好选择': 'chosen'},
 {'样本': 'P06',
  '训练前margin': -0.405,
  '训练后margin': 10.029,
  '跨过目标间隔': True,
  '偏好选择': 'chosen'}]


## 6. 失败案例与修正：使用 log probability 求和会惩罚长 chosen

假设 chosen 有四个概率 0.25 的合理 token，rejected 只有两个概率 0.20 的空泛 token。chosen 每 token 概率更高，但求和后累积四个负数，margin 反而变负；SimPO 的长度归一化平均值会给出正确排序。实际实现还必须用 attention mask 排除 padding。

In [6]:
length_case_chosen = torch.log(torch.tensor([0.25, 0.25, 0.25, 0.25], dtype=torch.float32))  # 构造四 token 优选回答的真实对数概率
length_case_rejected = torch.log(torch.tensor([0.20, 0.20], dtype=torch.float32))  # 构造两 token 拒选回答的真实对数概率
sum_margin = float(length_case_chosen.sum() - length_case_rejected.sum())  # 复现未归一化序列和对长回答的系统性惩罚
average_margin = float(length_case_chosen.mean() - length_case_rejected.mean())  # 用每 token 平均奖励消除纯长度偏置
padding_logp = torch.tensor([-1.0, -1.2, -9.0, -9.0], dtype=torch.float32)  # 构造含两个真实 token 和两个 padding 的批量序列
attention_mask = torch.tensor([1.0, 1.0, 0.0, 0.0], dtype=torch.float32)  # 标记只有前两个位置属于真实回答
masked_average = float((padding_logp * attention_mask).sum() / attention_mask.sum())  # 用显式 mask 计算不受 padding 影响的平均奖励
print({"失败_求和margin": round(sum_margin, 4), "修正_平均margin": round(average_margin, 4), "含padding的mask平均": round(masked_average, 4), "解释": "求和误判rejected，平均值正确偏好chosen"})  # 展示长度偏置与 padding 修正

{'失败_求和margin': -2.3263, '修正_平均margin': 0.2231, '含padding的mask平均': -1.1, '解释': '求和误判rejected，平均值正确偏好chosen'}


## 7. 生产差距与最小回归检查

真实 LLM 要按 assistant token mask 汇总自回归 log probability，并处理 packed sequence、EOS、截断和分布式 all-reduce。β、γ、学习率与回答长度分布需要联合调参，还要监控 reward hacking、风格单一化、KL 漂移和独立人工偏好胜率。reference-free 只是省掉参考模型，并不省掉 SFT 初始化与质量护栏。最后的断言只验证本实验已展示的梯度、收敛、逐样本 margin 和长度失败。

In [7]:
assert len(cases) >= 5  # 确认真实偏好对数量满足逐样本教学要求
assert first_gradient_norm > 0.0  # 确认 SimPO 损失真实产生了策略参数梯度
assert loss_history[-1] < loss_history[0]  # 确认优化过程降低了 reference-free 偏好损失
assert after_hits == len(cases)  # 确认训练后六组回答都正确偏好 chosen
assert all(row["跨过目标间隔"] for row in result_rows)  # 确认每组偏好都超过设定的 γ 间隔
assert sum_margin < 0.0 < average_margin  # 确认未归一化求和真实误判而长度平均修正排序
assert abs(masked_average + 1.1) < 1e-6  # 确认 padding 位置没有污染序列平均奖励
print("回归检查通过：SimPO 梯度、目标间隔、长度归一化与 padding mask 均已验证。")  # 输出最终验收结论

回归检查通过：SimPO 梯度、目标间隔、长度归一化与 padding mask 均已验证。
